In [27]:
import pandas as pd
import networkx as nx
import numpy as np

from collections import defaultdict

In [10]:
df_q_u  = pd.read_excel("Qwen_triples_unstructured.xlsx")
df_q_s  = pd.read_excel("Qwen_triples_structured.xlsx")
df_q_ss = pd.read_excel("Qwen_triples_semi-structured.xlsx")

df_g_ss = pd.read_excel("Gemma_triples_semi-structured.xlsx")

In [11]:
outputs = {"qwen" : {
            "unstructured" : df_q_u["triplets"],
            "structured" : df_q_s["triplets"],
            "semi-structured" : df_q_ss["triplets"]
           },
           "gemma" : {
               "semi-structured" : df_g_ss["triplets"]
           }
          }

In [43]:
def show_descriptives(outputs, model, prompt):
    full_output = outputs[model][prompt]

    vacancies = [eval(triple) for triple in full_output]

    stats = defaultdict(list)

    for v in vacancies:
        G = nx.DiGraph()
        
        for edge in v:
            if len(edge) == 3:
                a, relation, b = edge
                G.add_edge(a, b, relation=relation)
            else:
                pass

        stats["num nodes"].append(len(G.nodes()))
        stats["num edges"].append(len(G.edges()))
        stats["mean degree"].append(np.mean(list(dict(G.degree()).values())))
        stats["mean betweenness"].append(np.mean(list(dict(nx.betweenness_centrality(G)).values())))      
        stats["density"].append(nx.density(G))

    for k, v in stats.items():
        stats[k] = float(np.mean(v))
        
    return (model, prompt, stats)

for model in ["qwen", "gemma"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        if (model in outputs) and (prompt in outputs[model]):
            print(show_descriptives(outputs, model, prompt))
            print()

('qwen', 'structured', defaultdict(<class 'list'>, {'num nodes': 80.24, 'num edges': 74.24, 'mean degree': 1.82498264780216, 'mean betweenness': 2.313691929300386e-05, 'density': 0.01692419554193124}))

('qwen', 'semi-structured', defaultdict(<class 'list'>, {'num nodes': 100.72, 'num edges': 98.16, 'mean degree': 1.9245279874206818, 'mean betweenness': 3.7660448911503267e-06, 'density': 0.015493761223203695}))

('qwen', 'unstructured', defaultdict(<class 'list'>, {'num nodes': 142.2, 'num edges': 138.52, 'mean degree': 1.9083925027698405, 'mean betweenness': 1.1168741818896616e-08, 'density': 0.028287408745798794}))

('gemma', 'semi-structured', defaultdict(<class 'list'>, {'num nodes': 60.68, 'num edges': 56.84, 'mean degree': 1.857091192467225, 'mean betweenness': 0.0001344182685850404, 'density': 0.020448221390535343}))

